# 모델분석

좋은 모델을 구상하셨습니다. seq2seq 모델을 채용하셔서 기존 모델 대비해서 최초 1분 이후의 데이터를 더 의미있게 예측할 수 있는 코드임을 확인했습니다.

복잡한 기능을 수행하지만 간결하고 효율적으로 잘 짜여진 코드입니다. 일찍 알았더라면 분명이 적용했을 모델입니다. 

다만 제가 코드를 따라가기가 버거워서 (정리할겸, 공부할겸) 각 레이어별로 기능을 상세하게 정리를 해봤습니다.

- 기존모델: 180분 학습 후 15분을 한 번에 예측. 최초 1분 이후의 데이터의 예측이 크게 의미가 없을 확률이 있다.
- seq2seq모델: 한번에 1분씩 15번 예측. 1분마다 어텐션 스코어를 계산하기 때문에 매분마다 의미있는 예측이 가능. 

In [ ]:
#인코더 행렬 정의
self.encoder = nn.LSTM(
    input_size=input_size, hidden_size=hidden_size,
    num_layers=num_layers, batch_first=True,
    dropout=dropout if num_layers > 1 else 0
)
self.step_embedding = nn.Embedding(output_size, embed_dim)

#어텐션 가중치 행렬 정의
self.attn_Wh = nn.Linear(hidden_size, hidden_size, bias=False)
self.attn_Ws = nn.Linear(hidden_size, hidden_size, bias=False) #Ws out size = encoder out size
#어텐션 결과 벡터 정의
self.attn_V = nn.Linear(hidden_size, 1, bias=False) #Wh out size = V in size


# 인코더 행렬과 가중치 행렬
$$ 
   \text{Encoder}_{LSTM} \in M_{(10 \times 128)} \\\\
   W_h \in M_{(128 \times 128)} \\\\
   W_s \in M_{(128 \times 128)} \\\\
   V \in M_{(128 \times 1)}
$$
혹시 이 모델을 추후 재사용하시려면 위의 인코더, 가중치 행렬들 모양을 반드시 신경쓰셔야합니다.

지금은 가중치 행렬이 정사각형이기 때문에 괜찮은데, 혹시나 직사각형 행렬을 사용하시면 모양에 꼭 유의하셔야됩니다.

이 행렬들은 어텐션 에너지 (forward 함수 안에 있는 energy 변수)를 계산할 때 다음과 같이 쓰입니다.

$$ \text{Energy} = \tanh(W_h \cdot h_{\text{dec}} + W_s \cdot \text{enc\_outputs})$$

이때 괄호 안의 $W_h \cdot h_{\text{dec}} + W_s \cdot \text{enc\_outputs}$ 부분의 연산을 수행하려면 당연히 행렬 사이즈가 맞아야합니다.

그러려면 $W_h$가 받는 인풋사이즈가 $h_{\text{dec}}$의 사이즈 (=LSTM 인코더의 아웃풋 사이즈)와 같아야합니다 (그래야 둘을 곱할 수 있으니까요).

또한, $W_s$가 받는 인풋사이즈도 $\text{enc\_outputs}$의 사이즈 (=LSTM 인코더의 아웃풋 사이즈)와 같아야합니다.

마지막으로 두 행렬을 더하려면 같은 사이즈여야하니 $W_h, W_s$도 당연히 같은 모양이어야합니다. 정리하자면, 현재 윤영님이 짠 모델에서는 아래가 
반드시 성립해야합니다.

- $W_h$ in = LSTM encoder out
- $W_s$ in = LSTM encoder out
- $W_h$ shape = $W_s$ shape




**참고**

행렬곱은 수학에서 선형함수로도 많이 표현하기 때문에 $W_h \cdot h_{\text{dec}}$ 을 $W_h(h_{\text{dec}})$와 같이 많이 표현합니다.

실제 윤영님 코드에서도 행렬곱이 선형함수로 표현이 되어있고요. 

다만 비전공자한테 친숙하지 않은 표현법일 수 있어 행렬곱을 곱하기 기호 $(\cdot)$로 표현할게요.

In [ ]:
energy = torch.tanh(query + enc_keys_projected)
score = self.attn_V(energy).squeeze(-1)
attn_weights = torch.softmax(score, dim=1)

# Energy 계산 후 Score 계산

스코어는 어텐션 벡터에 에너지를 곱해서 계산합니다.
$$\text{Score} = V \cdot \text{Energy}$$
역시 모양이 맞아야겠죠? Energy행렬의 마지막 axis의 모양이 $V$의 인풋사이즈와 같아야합니다. 

Energy.shape = [batch_size, window_size=180 , hidden_size=128] 이고

V의 인풋사이즈는 128이니 연산에 문제가 없습니다.

V의 아웃풋사이즈는 1입니다. 따라서 score는 [batch_size, window_size=180, 1] 의 모양입니다.

이후 score에 softmax를 취해서 t-분을 예측할 때 사용할 어텐션 가중치를 결정합니다 (즉, t-분째를 예측할 때 180분 중에서 어떤 '분'이 가장 중요한지 결정합니다. sofxmax값이 있는 인덱스(분)을 취하겠죠).

In [ ]:
self.decoder = nn.LSTMCell(
    input_size=hidden_size + embed_dim, hidden_size=hidden_size
)
self.layer_norm = nn.LayerNorm(hidden_size)
self.fc_out = nn.Linear(hidden_size, 1)

context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)
step_emb = step_embs[t].unsqueeze(0).expand(batch_size, -1)
dec_input = torch.cat([context, step_emb], dim=1)
h_dec, c_dec = self.decoder(dec_input, (h_dec, c_dec))
pred_t = self.fc_out(self.layer_norm(h_dec))
predictions.append(pred_t)

# 임베딩 레이어와 디코딩 레이어

step_embedding은 현재 '분'을 decoder에게 알려주기 위해 추가한 특성입니다.

기존 15분을 한번에 예측하던걸 1분씩 루프를 돌면서 예측을 하는데 이 1분마다에 순서성을 부여해야되기 때문에 step_embedding을 해서 순서성을 임시적인 특성으로 중간결과값에 추가합니다.

이 특성이 포함된 중간결과값을 decoder가 해석합니다 (그래서 decoder 인풋은  hidden + embed 입니다).

해석한 결과로 decoder는 히든스테이트(h_dec)와 셀스테이트(c_dec)을 리턴합니다.

최종적으로 히든은 해당 분에 직접적인 아웃풋으로 변환되고 (layer_norm), 셀은 다음 for loop에서 재사용됩니다.

In [ ]:
"""
seq2seq_predictor.py - 배수지 13(j_resv) 전용 Seq2Seq+Attention 모델
"""

import torch
import torch.nn as nn


class LSTMSeq2SeqAttnModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2,
                 output_size=15, embed_dim=16, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        #인코더 행렬 정의
        self.encoder = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        #임베딩 레이어 정의
        self.step_embedding = nn.Embedding(output_size, embed_dim)
        
        #어텐션 가중치 행렬 정의
        self.attn_Wh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attn_Ws = nn.Linear(hidden_size, hidden_size, bias=False) #Ws out size = encoder out size
        #어텐션 결과 벡터 정의
        self.attn_V = nn.Linear(hidden_size, 1, bias=False) #Wh out size = V in size
        #디코딩 레이어
        self.decoder = nn.LSTMCell(
            input_size=hidden_size + embed_dim, hidden_size=hidden_size
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc_out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        batch_size = x.size(0)
        enc_outputs, (h_n, c_n) = self.encoder(x)
        enc_keys_projected = self.attn_Ws(enc_outputs)
        h_dec = h_n[-1]
        c_dec = c_n[-1]
        predictions = []
        step_ids = torch.arange(self.output_size, device=x.device)
        step_embs = self.step_embedding(step_ids)

        for t in range(self.output_size):
            query = self.attn_Wh(h_dec).unsqueeze(1)
            #어텐션 점수 계산
            energy = torch.tanh(query + enc_keys_projected)
            score = self.attn_V(energy).squeeze(-1)
            attn_weights = torch.softmax(score, dim=1)
            #bmm = batch matrix multiplication
            #배치사이즈 개수의 행렬들을 곱함
            context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)
            #임베딩 적용후 디코딩
            step_emb = step_embs[t].unsqueeze(0).expand(batch_size, -1)
            dec_input = torch.cat([context, step_emb], dim=1)
            h_dec, c_dec = self.decoder(dec_input, (h_dec, c_dec))
            pred_t = self.fc_out(self.layer_norm(h_dec))
            predictions.append(pred_t)

        return torch.cat(predictions, dim=1)
